In [2]:
import os
import numpy as np
import utm

class OxtData:
    def __init__(self, lat, lon, alt, roll, pitch, yaw, pos_accuracy):
        self.lat = lat
        self.lon = lon
        self.alt = alt
        self.roll = roll
        self.pitch = pitch
        self.yaw = yaw
        self.pos_accuracy = pos_accuracy

def read_oxt_file(file_path):
    with open(file_path, 'r') as file:
        line = file.readline()
        data = list(map(float, line.split()))
        # Assuming the order of data is: lat, lon, alt, roll, pitch, yaw, pos_accuracy
        oxt_data = OxtData(*data[:7])
    return oxt_data

def convert_to_utm(oxt_data):
    # Convert latitude and longitude to UTM coordinates
    x, y, zone_number, zone_letter = utm.from_latlon(oxt_data.lat, oxt_data.lon)
    return x, y, zone_number, zone_letter

def calculate_distance(p1, p2):
    # Calculate Euclidean distance between two points
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def process_oxts_folder(folder_path, start_index=22, distance_threshold=2.0):
    utm_pose = []
    last_utm = None
    utm_to_info = {}

    # Sort the files by their numerical part
    files = sorted([f for f in os.listdir(folder_path) if f.endswith('.txt')], key=lambda x: int(x.split('.')[0]))

    for i, filename in enumerate(files):
        if i < start_index:
            continue
        
        file_path = os.path.join(folder_path, filename)
        oxt_data = read_oxt_file(file_path)
        x, y, zone_number, zone_letter = convert_to_utm(oxt_data)
        
        if last_utm is None or calculate_distance(last_utm, (x, y)) >= distance_threshold:
            utm_pose.append((x, y, zone_number, zone_letter))
            last_utm = (x, y)
            # Record the UTM coordinates, latitude, longitude, and filename
            utm_to_info[(x, y, zone_number, zone_letter)] = {
                'lat': oxt_data.lat,
                'lon': oxt_data.lon,
                'filename': filename
            }

    return utm_pose, utm_to_info

def calculate_relative_positions(utm_pose, num_points=15):
    trajectory_ins = {}
    
    for i, current_point in enumerate(utm_pose):
        if i + num_points > len(utm_pose):
            # If there are fewer than num_points points after the current point, skip it
            continue
        
        relative_positions = []
        for j in range(i, i + num_points):
            next_point = utm_pose[j]
            dx = next_point[0] - current_point[0]
            dy = next_point[1] - current_point[1]
            relative_positions.append((-dx, -dy))  # 注意转换坐标系
        
        # Store the relative positions in the dictionary
        trajectory_ins[current_point] = relative_positions
    
    return trajectory_ins

def calculate_past_relative_positions(utm_pose, num_points=5):
    trajectory_ins_past = {}
    
    for i, current_point in enumerate(utm_pose):
        if i < num_points:
            # If there are fewer than num_points points before the current point, skip it
            continue
        
        relative_positions = []
        for j in range(i - 1, max(i - num_points - 1, -1), -1):
            prev_point = utm_pose[j]
            dx = current_point[0] - prev_point[0]
            dy = current_point[1] - prev_point[1]
            relative_positions.append((-dx, -dy)) # 注意转换坐标系
        
        # Store the relative positions in the dictionary
        trajectory_ins_past[current_point] = relative_positions
    
    return trajectory_ins_past

# Path to the OXTS data folder
folder_path = '/home/users/xzh/trajectory-prediction/datasets/my_test10/oxts/data'

# Process all OXTS files in the folder starting from the 22nd file
utm_pose, utm_to_info = process_oxts_folder(folder_path)

# Function to get the latitude, longitude, and filename for a given UTM coordinate
def get_info_for_utm(utm_coord, utm_to_info):
    return utm_to_info.get(utm_coord, None)

# Example usage
utm_coord = utm_pose[0]  # Example UTM coordinate
info = get_info_for_utm(utm_coord, utm_to_info)
if info:
    print(f"UTM Coordinate: {utm_coord}")
    print(f"Latitude: {info['lat']}, Longitude: {info['lon']}, Filename: {info['filename']}")
else:
    print("UTM coordinate not found.")

# Calculate the relative positions for each point in the UTM coordinates
trajectory_ins = calculate_relative_positions(utm_pose)

# # Optionally, print out the trajectory_ins dictionary
# for point, rel_positions in trajectory_ins.items():
#     print(f"Point: x={point[0]}, y={point[1]}, Zone Number={point[2]}, Zone Letter={point[3]}")
#     for dx, dy in rel_positions:
#         print(f"\tRelative Position: dx={dx}, dy={dy}")

# Calculate the past relative positions for each point in the UTM coordinates
trajectory_ins_past = calculate_past_relative_positions(utm_pose)

# Optionally, print out the trajectory_ins_past dictionary
# for point, rel_positions in trajectory_ins_past.items():
#     print(f"Point: x={point[0]}, y={point[1]}")
#     for dx, dy in rel_positions:
#         print(f"\tRelative Position: dx={dx}, dy={dy}")

UTM Coordinate: (461846.6285936256, 5424529.115667883, 32, 'U')
Latitude: 48.972494784142, Longitude: 8.4786555328752, Filename: 0000000022.txt
